In [ ]:
# ============ 模块0 依赖检查（原 Kaggle 模板 cell 已移除）============
# Kaggle 环境自带这些库；本地首次运行前请先执行：
#   pip install numpy pandas torch nibabel matplotlib scikit-learn
try:
    import numpy
    import pandas
    import torch
    import nibabel
    import matplotlib
    print("依赖检查通过：numpy / pandas / torch / nibabel / matplotlib 均已安装")
except ImportError as e:
    print("缺少依赖:", e)
    print("请在终端执行: pip install numpy pandas torch nibabel matplotlib scikit-learn")


In [ ]:
# %% ========================= 模块1 Environment =========================
import os
import glob
import random
from functools import lru_cache
import torch.nn as nn
 
import numpy as np
import nibabel as nib          # 若Kaggle环境未预装，取消下一行注释安装
# !pip install nibabel
 
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
 
 
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
 
 
SEED = 42
set_seed(SEED)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
 
# ============ 环境自适应配置：同一份代码，Kaggle / 本地都能跑 ============
IS_KAGGLE = os.path.exists("/kaggle/input") or os.path.exists("/kaggle/working")
 
if IS_KAGGLE:
    # ---- KiTS19 数据目录：优先用写死路径；不存在则自动探测（防挂载路径不一致）----
    DATA_ROOT = "/kaggle/input/datasets/user123454321/kits19-1"
    if not os.path.isdir(DATA_ROOT):
        _found_root = None
        for _input_dir in sorted(glob.glob("/kaggle/input/*")):
            _roots = [_input_dir] + [
                os.path.join(_input_dir, e)
                for e in sorted(os.listdir(_input_dir))
                if os.path.isdir(os.path.join(_input_dir, e))
            ]
            for _r in _roots:
                _case_dirs = [
                    e for e in os.listdir(_r)
                    if e.startswith("case_") and os.path.isdir(os.path.join(_r, e))
                ]
                if _case_dirs:
                    _found_root = _r
                    break
            if _found_root:
                break
        if _found_root is None:
            raise FileNotFoundError(
                "未找到 KiTS19 数据集：请先在 Kaggle 右侧 Add Dataset 挂载含 case_* 的数据集，"
                "或手动修改本 cell 的 DATA_ROOT"
            )
        DATA_ROOT = _found_root
        print("自动探测到 KiTS19 数据集: %s（病例数 %d）" % (DATA_ROOT, len(_case_dirs)))
    OUTPUT_DIR = "/kaggle/working"
    MAX_CASES  = None                        # Kaggle：全部病例
    NUM_WORKERS = 2
    EPOCHS     = 50
    BATCH_SIZE = 8
    TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
else:
    # ===== 本地快速验证模式（只验证逻辑，参数很小）=====
    DATA_ROOT = r"E:\大创\kits19_small"    # <-- 改成本地放小数据集的文件夹
    OUTPUT_DIR = r"E:\大创\output"         # 本地结果输出目录
    MAX_CASES  = 4                           # 本地快速验证：取前 4 个病例
    NUM_WORKERS = 0                          # Windows 本地必须为 0
    EPOCHS     = 1                           # 本地只跑 1 个 epoch
    BATCH_SIZE = 2
    TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.5, 0.25, 0.25   # 本地 4 病例：2 训练 / 1 验证 / 1 测试
 
MC_N_SAMPLES = int(os.environ.get("MC_N_SAMPLES", "10"))  # 贝叶斯 MC 采样数（人员3），可按需调小提速
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"运行环境: {'Kaggle' if IS_KAGGLE else '本地'}")
print(f"数据路径: {DATA_ROOT}")
print(f"输出目录: {OUTPUT_DIR}")


In [ ]:
# %% ========================= 模块1.5 仓库与实验准备（自适应，人员2）=========================
# 一次运行跑三个实验（data10pct / data20pct / datafull）。本 cell 负责：
#   A. 仓库准备：Kaggle 自动 git clone（GitHub 导入只带 notebook，不带其余代码文件）；本地跳过；
#   B. 小样本 manifest：确保 experiments/data10pct|data20pct/manifest.json 存在（缺失则自动生成）；
#      datafull 的 manifest 由模块4/4.5 从全量划分写入。
import subprocess
import sys

# ---- A. 仓库准备（仅 Kaggle 需要；本地当前目录即仓库，跳过）----
if IS_KAGGLE:
    repo_dir = "/kaggle/working/code"
    if os.path.isdir(os.path.join(repo_dir, ".git")):
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], capture_output=True, text=True)
        print("Kaggle：仓库已存在，已 git pull 更新")
    else:
        r = subprocess.run(
            ["git", "clone", "https://github.com/yijunv/code.git", repo_dir],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            raise RuntimeError("git clone 失败（请先确认 notebook 设置里 Internet 已开启）：\n" + r.stderr)
        print("Kaggle：仓库 clone 完成")
    os.chdir(repo_dir)
    print("仓库根目录:", sorted(os.listdir(".")))
else:
    print("本地模式：跳过仓库 clone（当前目录即仓库）")

# ---- B. 确保小样本 manifest（10%/20%）存在 ----
EXPERIMENTS_ROOT = os.path.join(OUTPUT_DIR, "experiments")
os.makedirs(EXPERIMENTS_ROOT, exist_ok=True)
_missing = [n for n in ("data10pct", "data20pct")
            if not os.path.isfile(os.path.join(EXPERIMENTS_ROOT, n, "manifest.json"))]
if _missing:
    downsample_py = os.path.join(os.getcwd(), "downsample.py")
    if not os.path.isfile(downsample_py):
        raise FileNotFoundError("找不到 downsample.py: %s" % downsample_py)
    cmd = [sys.executable, downsample_py,
           "--data-root", DATA_ROOT,
           "--out-dir", EXPERIMENTS_ROOT,
           "--fractions", "0.10,0.20",
           "--seed", str(SEED)]
    if not IS_KAGGLE:  # 本地验证用小数据集自身划分
        cmd += ["--train-ratio", str(TRAIN_RATIO), "--val-ratio", str(VAL_RATIO),
                "--max-cases", str(MAX_CASES)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("downsample.py 执行失败：\n" + r.stdout + r.stderr)
    print(r.stdout)
    print("已生成缺失的 manifest:", _missing)
else:
    print("小样本 manifest 已存在:", os.listdir(EXPERIMENTS_ROOT))

print("实验根目录:", EXPERIMENTS_ROOT)


In [ ]:
# %% ========================= 模块2 Config =========================
class Config:
    # 数据路径（来自模块1的环境自适应配置）
    DATA_ROOT = DATA_ROOT
    OUTPUT_DIR = OUTPUT_DIR
 
    # 数据集划分比例（按病例划分，避免同一病例的切片同时出现在train/val/test）
    TRAIN_RATIO = TRAIN_RATIO
    VAL_RATIO = VAL_RATIO
    TEST_RATIO = TEST_RATIO
 
    # 本地快速验证时只取前几个病例
    MAX_CASES = MAX_CASES
    NUM_WORKERS = NUM_WORKERS
 
    # 图像与标签
    IMG_SIZE = 256
    NUM_CLASSES = 3          # 0=背景, 1=肾脏, 2=肿瘤
    HU_MIN = -200            # CT窗宽窗位裁剪下界
    HU_MAX = 300              # CT窗宽窗位裁剪上界
 
    # 训练超参数（本地小数据模式已在模块1中调小）
    BATCH_SIZE = BATCH_SIZE
    LEARNING_RATE = 1e-4
    EPOCHS = EPOCHS
    # 训练集 DROP_LAST（小样本实验模式由模块4.5 改为 False，避免 drop 掉唯一批次）
    DROP_LAST_TRAIN = True
 
    # 预留给后续贝叶斯/不确定性模块，基础U-Net阶段暂不使用
    DROPOUT_RATE = 0.0
    UNCERTAINTY_THRESHOLD = None
 
    MC_N_SAMPLES = MC_N_SAMPLES
    SEED = SEED
    DEVICE = DEVICE
 
 
cfg = Config()


In [ ]:
# %% ========================= 模块3 Dataset =========================
def _find_volume(case_dir: str, name: str):
    """在病例目录中查找 imaging/segmentation 文件，兼容 .nii 和 .nii.gz"""
    for ext in (".nii", ".nii.gz"):
        vol = os.path.join(case_dir, name + ext)
        if os.path.isfile(vol):
            return vol
    return None


def scan_valid_cases(data_root: str, max_cases: int = None):
    """扫描data_root下所有case_*目录，保留同时具有imaging和segmentation的病例（兼容.nii/.nii.gz）"""
    case_dirs = sorted(glob.glob(os.path.join(data_root, "case_*")))
    valid_cases = [
        c for c in case_dirs
        if _find_volume(c, "imaging") is not None
        and _find_volume(c, "segmentation") is not None
    ]
    if max_cases is not None and len(valid_cases) > max_cases:
        valid_cases = valid_cases[:max_cases]
    print(f"数据集扫描完成: 共 {len(valid_cases)} 个有效病例（来自 {len(case_dirs)} 个目录）")
    return valid_cases
 
 
class KiTS19SliceDataset(Dataset):
    """
    从KiTS19病例的3D CT体积中提取2D轴向切片（假定数组形状为 [切片数, H, W]）。
    只保留含有肾脏(1)或肿瘤(2)标注的切片，过滤纯背景切片。
 
    初始化时一次性读盘、完成窗宽窗位归一化与resize，并把结果常驻内存；
    __getitem__ 只从内存取数据，不再重复读盘。
    （此前每次__getitem__现读整卷CT、缓存位只有4个，在shuffle训练下几乎每张切片
    都要重新读一次约1.2GB的体数据，是导致训练卡住不动的根本原因，这里做了修正）
    """
 
    def __init__(self, case_dirs, img_size, hu_min, hu_max, transform=None):
        self.img_size = img_size
        self.transform = transform
        self.images = []  # 每个元素: (img_size, img_size) float32
        self.masks = []   # 每个元素: (img_size, img_size) int64
        self._preload(case_dirs, hu_min, hu_max)
 
    def _preload(self, case_dirs, hu_min, hu_max):
        for case_dir in case_dirs:
            img_vol = nib.load(_find_volume(case_dir, "imaging")).get_fdata()
            seg_vol = nib.load(_find_volume(case_dir, "segmentation")).get_fdata()
            nonempty_idx = np.where(seg_vol.sum(axis=(1, 2)) > 0)[0]
 
            for idx in nonempty_idx:
                img = img_vol[idx, :, :].astype(np.float32)
                mask = seg_vol[idx, :, :].astype(np.int64)
 
                img = np.clip(img, hu_min, hu_max)
                img = (img - hu_min) / (hu_max - hu_min)
 
                img_t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
                mask_t = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0).float()
 
                img_t = F.interpolate(img_t, size=(self.img_size, self.img_size),
                                       mode="bilinear", align_corners=False)
                mask_t = F.interpolate(mask_t, size=(self.img_size, self.img_size), mode="nearest")
 
                self.images.append(img_t.squeeze(0).squeeze(0).numpy().astype(np.float32))
                self.masks.append(mask_t.squeeze(0).squeeze(0).numpy().astype(np.int64))
 
        print(f"切片预加载完成: 共 {len(self.images)} 个有效切片（已过滤纯背景切片，已常驻内存）")
 
    def __len__(self):
        return len(self.images)
 
    def __getitem__(self, i):
        img_t = torch.from_numpy(self.images[i]).unsqueeze(0)  # (1, H, W)
        mask_t = torch.from_numpy(self.masks[i])                # (H, W)
 
        if self.transform is not None:
            img_t, mask_t = self.transform(img_t, mask_t)
 
        return img_t, mask_t


In [ ]:
# %% ========================= 模块4 DataLoader（全量划分，datafull 基准）=========================
def split_cases(case_dirs, train_ratio, val_ratio, seed):
    """按病例（而非切片）划分train/val/test，避免同一病例的切片跨集合造成数据泄漏"""
    rng = random.Random(seed)
    cases = case_dirs.copy()
    rng.shuffle(cases)
    n = len(cases)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_cases = cases[:n_train]
    val_cases = cases[n_train:n_train + n_val]
    test_cases = cases[n_train + n_val:]
    return train_cases, val_cases, test_cases


all_cases = scan_valid_cases(cfg.DATA_ROOT, cfg.MAX_CASES)
train_cases, val_cases, test_cases = split_cases(
    all_cases, cfg.TRAIN_RATIO, cfg.VAL_RATIO, cfg.SEED
)
print("全量划分（datafull 基准）: 训练 %d 例 | 验证 %d 例 | 测试 %d 例（共 %d 例）" % (
    len(train_cases), len(val_cases), len(test_cases), len(all_cases)))
# 数据集/loader 不在此构建：由模块4.5 的 setup_experiment() 按各实验的 manifest 构建


In [ ]:
# %% ============ 模块4.5 实验准备：datafull manifest + setup_experiment（M5 小样本，人员2） ============
# setup_experiment(exp_name) 按 manifest 构建该实验的数据集/loader，并把 cfg.OUTPUT_DIR 切到实验目录，
# 保证 manifest.json / best_unet.pth / 训练曲线 / evaluation_metrics.csv / metrics.json / samples/ 全部落在同一实验目录内。
import json


def _write_full_manifest():
    """datafull：用与模块4 相同口径重新做全量划分并写 manifest.json（契约 C）。

    不依赖外层运行时变量：模块13 的循环会把 train_cases 等变量改成当前实验的值，
    这里在函数内独立重算全量划分（同 seed 同比例，结果与模块4 打印一致）。
    """
    all_cases_full = scan_valid_cases(cfg.DATA_ROOT, cfg.MAX_CASES)
    full_train, full_val, full_test = split_cases(
        all_cases_full, cfg.TRAIN_RATIO, cfg.VAL_RATIO, cfg.SEED
    )
    full_dir = os.path.join(EXPERIMENTS_ROOT, "datafull")
    os.makedirs(full_dir, exist_ok=True)
    manifest = {
        "experiment": "datafull",
        "data_root": cfg.DATA_ROOT,
        "img_size": cfg.IMG_SIZE,
        "seed": cfg.SEED,
        "data_fraction": 1.0,
        "train_ratio": cfg.TRAIN_RATIO,
        "val_ratio": cfg.VAL_RATIO,
        "test_ratio": round(1.0 - cfg.TRAIN_RATIO - cfg.VAL_RATIO, 4),
        "max_cases": cfg.MAX_CASES,
        "n_total_cases": len(all_cases_full),
        "n_train_cases": len(full_train),
        "n_val_cases": len(full_val),
        "n_test_cases": len(full_test),
        "train_cases": [os.path.basename(p) for p in full_train],
        "val_cases": [os.path.basename(p) for p in full_val],
        "test_cases": [os.path.basename(p) for p in full_test],
        "notes": "datafull 为全量基准实验：与 downsample.py 同 seed 同口径的全量划分（测试集冻结）。",
    }
    path = os.path.join(full_dir, "manifest.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return path

def setup_experiment(exp_name, train_transform=None):
    """按 manifest 构建指定实验的数据集/loader，并把产物输出目录切到该实验目录。返回 dict 环境。"""
    if exp_name == "datafull":
        _write_full_manifest()
    manifest_path = os.path.join(EXPERIMENTS_ROOT, exp_name, "manifest.json")
    if not os.path.isfile(manifest_path):
        raise FileNotFoundError("找不到 manifest: %s" % manifest_path)
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    def _cases_from_manifest(key):
        return [os.path.join(cfg.DATA_ROOT, name) for name in manifest[key]]

    tr_cases = _cases_from_manifest("train_cases")
    va_cases = _cases_from_manifest("val_cases")
    te_cases = _cases_from_manifest("test_cases")

    cfg.EXPERIMENT_MANIFEST = manifest_path
    cfg.EXPERIMENT_DIR = os.path.dirname(manifest_path)
    cfg.OUTPUT_DIR = cfg.EXPERIMENT_DIR
    cfg.DROP_LAST_TRAIN = False
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

    train_dataset = KiTS19SliceDataset(tr_cases, cfg.IMG_SIZE, cfg.HU_MIN, cfg.HU_MAX, transform=None)
    val_dataset = KiTS19SliceDataset(va_cases, cfg.IMG_SIZE, cfg.HU_MIN, cfg.HU_MAX, transform=None)
    test_dataset = KiTS19SliceDataset(te_cases, cfg.IMG_SIZE, cfg.HU_MIN, cfg.HU_MAX, transform=None)
    if train_transform is not None:
        train_dataset.transform = train_transform

    train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
                              num_workers=cfg.NUM_WORKERS, drop_last=cfg.DROP_LAST_TRAIN)
    val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS)

    print("=" * 50)
    print("实验 %s 数据就绪 | 训练 %d 例 | 验证 %d 例 | 测试 %d 例" % (
        exp_name, len(tr_cases), len(va_cases), len(te_cases)))
    print("产物输出目录: %s" % cfg.OUTPUT_DIR)
    print("=" * 50)
    return {
        "train_cases": tr_cases, "val_cases": va_cases, "test_cases": te_cases,
        "train_dataset": train_dataset, "val_dataset": val_dataset, "test_dataset": test_dataset,
        "train_loader": train_loader, "val_loader": val_loader, "test_loader": test_loader,
    }


In [ ]:
# %% ========================= 模块5 Augmentation（定义，由模块4.5 挂载到训练集）=========================
class TrainAugmentation:
    """
    训练阶段增强：随机水平翻转 / 随机垂直翻转 / 随机90度旋转，图像与mask同步变换。
    验证/测试阶段不做随机增强，只保留模块3中已完成的窗宽窗位归一化与resize。
    """
 
    def __call__(self, img, mask):
        if random.random() < 0.5:
            img = torch.flip(img, dims=[-1])
            mask = torch.flip(mask, dims=[-1])
        if random.random() < 0.5:
            img = torch.flip(img, dims=[-2])
            mask = torch.flip(mask, dims=[-2])
        if random.random() < 0.5:
            k = random.choice([1, 2, 3])
            img = torch.rot90(img, k, dims=[-2, -1])
            mask = torch.rot90(mask, k, dims=[-2, -1])
        return img, mask
 
 
train_transform = TrainAugmentation()
print("数据增强已定义（仅训练集使用）: 随机水平/垂直翻转 + 随机90°旋转")


In [ ]:
# %% ========================= 模块6 Model =========================
class DoubleConv(nn.Module):
    """(Conv3x3 -> BN -> ReLU) x2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """基础U-Net，输入单通道CT切片，输出num_classes通道的分割logits（不含贝叶斯/Dropout采样机制）"""
    def __init__(self, in_channels=1, num_classes=3, base_channels=32):
        super().__init__()
        c1, c2, c3, c4, c5 = base_channels, base_channels * 2, base_channels * 4, base_channels * 8, base_channels * 16
        self.enc1 = DoubleConv(in_channels, c1)
        self.enc2 = DoubleConv(c1, c2)
        self.enc3 = DoubleConv(c2, c3)
        self.enc4 = DoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c4, c5)
        self.up4 = nn.ConvTranspose2d(c5, c4, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(c5, c4)
        self.up3 = nn.ConvTranspose2d(c4, c3, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(c4, c3)
        self.up2 = nn.ConvTranspose2d(c3, c2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(c3, c2)
        self.up1 = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(c2, c1)
        self.out_conv = nn.Conv2d(c1, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)  # (B, num_classes, H, W) logits


def build_model(cfg):
    """每个实验创建独立的模型实例。"""
    model = UNet(in_channels=1, num_classes=cfg.NUM_CLASSES, base_channels=32).to(cfg.DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"模型已构建: 基础U-Net | 参数量: {n_params:,}")
    return model


In [ ]:
# %% ========================= 模块7 Loss & Optimizer =========================
class SoftDiceLoss(nn.Module):
    """多类别可微Dice损失，对logits做softmax后与one-hot标签计算Dice，各类别取平均"""
    def __init__(self, num_classes, eps=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.eps = eps
    def forward(self, logits, target):
        probs = torch.softmax(logits, dim=1)
        target_onehot = F.one_hot(target, num_classes=self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = torch.sum(probs * target_onehot, dims)
        union = torch.sum(probs + target_onehot, dims)
        dice_per_class = (2 * intersection + self.eps) / (union + self.eps)
        return 1.0 - dice_per_class.mean()


class CombinedLoss(nn.Module):
    """CrossEntropy + Dice 组合损失（等权重）。多类别分割用CE替代原方案中的BCE"""
    def __init__(self, num_classes):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = SoftDiceLoss(num_classes)
    def forward(self, logits, target):
        return self.ce(logits, target) + self.dice(logits, target)


def build_optimizer(model, cfg):
    """每个实验创建独立的损失/优化器/调度器（绑定该实验的模型）。"""
    criterion = CombinedLoss(cfg.NUM_CLASSES)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
    print(f"损失函数: CrossEntropy + Dice | 优化器: Adam(lr={cfg.LEARNING_RATE}) | 调度器: ReduceLROnPlateau")
    return criterion, optimizer, scheduler


In [ ]:

# %% ========================= 模块8 Training =========================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
 
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item() * imgs.size(0)
 
    return running_loss / len(loader.dataset)

In [ ]:
# %% ========================= 模块9 Validation + 训练循环函数 =========================
def compute_dice_iou(pred, target, num_classes, eps=1e-6):
    """基于argmax硬预测计算各类别Dice和IoU，返回按类别平均值（含背景）"""
    dice_scores, iou_scores = [], []
    for c in range(num_classes):
        pred_c = (pred == c)
        target_c = (target == c)
        intersection = (pred_c & target_c).sum().item()
        pred_sum = pred_c.sum().item()
        target_sum = target_c.sum().item()
        union = pred_sum + target_sum - intersection
        if target_sum == 0 and pred_sum == 0:
            dice_scores.append(1.0)
            iou_scores.append(1.0)
        else:
            dice_scores.append((2 * intersection + eps) / (pred_sum + target_sum + eps))
            iou_scores.append((intersection + eps) / (union + eps))
    return float(np.mean(dice_scores)), float(np.mean(iou_scores))


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes):
    model.eval()
    running_loss, dice_sum, iou_sum, n_batches = 0.0, 0.0, 0.0, 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        loss = criterion(logits, masks)
        preds = torch.argmax(logits, dim=1)
        dice, iou = compute_dice_iou(preds, masks, num_classes)
        running_loss += loss.item() * imgs.size(0)
        dice_sum += dice
        iou_sum += iou
        n_batches += 1
    return {
        "loss": running_loss / len(loader.dataset),
        "dice": dice_sum / n_batches,
        "iou": iou_sum / n_batches,
    }


def train_experiment(cfg, model, train_loader, val_loader, criterion, optimizer, scheduler):
    """执行一个实验的完整训练+验证循环，保存 best_unet.pth，返回 (history, best_val_dice, best_ckpt_path)。"""
    history = {"train_loss": [], "val_loss": [], "val_dice": [], "val_iou": []}
    best_val_dice = -1.0
    best_ckpt_path = os.path.join(cfg.OUTPUT_DIR, "best_unet.pth")

    print("=" * 50)
    print("开始训练: %s | epochs=%d | 输出=%s" % (os.path.basename(cfg.OUTPUT_DIR), cfg.EPOCHS, cfg.OUTPUT_DIR))
    print("=" * 50)
    for epoch in range(1, cfg.EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, cfg.DEVICE)
        history["train_loss"].append(train_loss)
        if len(val_loader) > 0:
            val_metrics = evaluate(model, val_loader, criterion, cfg.DEVICE, cfg.NUM_CLASSES)
            scheduler.step(val_metrics["loss"])
            history["val_loss"].append(val_metrics["loss"])
            history["val_dice"].append(val_metrics["dice"])
            history["val_iou"].append(val_metrics["iou"])
            print(f"Epoch [{epoch}/{cfg.EPOCHS}] | Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_metrics['loss']:.4f} | Val Dice: {val_metrics['dice']:.4f} | "
                  f"Val IoU: {val_metrics['iou']:.4f}")
            if val_metrics["dice"] > best_val_dice:
                best_val_dice = val_metrics["dice"]
                torch.save(model.state_dict(), best_ckpt_path)
                print(f"  -> 保存最优模型 (Val Dice: {best_val_dice:.4f})")
        else:
            print(f"Epoch [{epoch}/{cfg.EPOCHS}] | Train Loss: {train_loss:.4f} | (验证集为空，跳过验证)")
            if best_val_dice < 0.0:
                best_val_dice = 0.0
                torch.save(model.state_dict(), best_ckpt_path)
                print(f"  -> 验证集为空，保存当前轮模型: {best_ckpt_path}")
    print("=" * 50)
    if history["val_dice"]:
        print(f"训练完成，最优验证Dice: {best_val_dice:.4f}，模型已保存至: {best_ckpt_path}")
    else:
        print(f"训练完成（验证集为空，按最后轮保存），模型已保存至: {best_ckpt_path}")
    return history, best_val_dice, best_ckpt_path


In [ ]:
# %% ========================= 模块9.5 M1 贝叶斯推理管线（人员3 交付，适配新产物规范） =========================
import json
import numpy as np
import os


@torch.no_grad()
def mc_predict(model, img, n_samples=10, device=cfg.DEVICE):
    """
    贝叶斯 MC-Dropout 推理
    Returns:
        mean_pred: (H, W) 平均概率的 argmax 类别索引
        variance: (H, W) 逐像素方差（跨类别平均）
        entropy: (H, W) 逐像素熵
        tumor_avg_entropy: float 肿瘤区域平均熵
        pred_binary: (H, W) 二值预测掩码（肿瘤=1，非肿瘤=0）
    """
    model.train()
    img = img.unsqueeze(0).to(device)
    logits_stack = []
    for _ in range(n_samples):
        logits = model(img)
        logits_stack.append(logits.unsqueeze(0))
    logits_stack = torch.cat(logits_stack, dim=0)
    probs = torch.softmax(logits_stack, dim=2)
    mean_probs = probs.mean(dim=0).squeeze(0)
    mean_pred = torch.argmax(mean_probs, dim=0)
    pred_binary = (mean_pred == 2).float().cpu().numpy()
    variance = probs.var(dim=0).squeeze(0).mean(dim=0)
    epsilon = 1e-8
    entropy_per_pixel = -torch.sum(mean_probs * torch.log(mean_probs + epsilon), dim=0)
    tumor_mask = (mean_pred == 2).float()
    tumor_entropy_sum = (entropy_per_pixel * tumor_mask).sum()
    tumor_count = tumor_mask.sum() + epsilon
    tumor_avg_entropy = (tumor_entropy_sum / tumor_count).item()
    return (
        mean_pred.cpu().numpy(),
        variance.cpu().numpy(),
        entropy_per_pixel.cpu().numpy(),
        tumor_avg_entropy,
        pred_binary,
    )


def save_mc_predict_outputs(model, dataset, output_dir, n_samples=10, device=cfg.DEVICE):
    """
    人员3 交付（适配新产物规范）：每个样本一个 sample_XXXX.npz + sample_XXXX.json，落在 samples/ 下。
    npz 字段: mean / variance / entropy / pred / label；json 为该样本元信息。
    """
    os.makedirs(output_dir, exist_ok=True)
    for idx in range(len(dataset)):
        img, mask = dataset[idx]
        mean_pred, variance, entropy, tumor_avg_entropy, pred_binary = mc_predict(
            model, img, n_samples=n_samples, device=device
        )
        sid = idx + 1
        save_path = os.path.join(output_dir, "sample_%04d.npz" % sid)
        np.savez(save_path,
                 mean=mean_pred, variance=variance, entropy=entropy,
                 pred=pred_binary, label=mask.numpy())
        meta_path = os.path.join(output_dir, "sample_%04d.json" % sid)
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump({
                "sample_id": sid,
                "dataset_index": idx,
                "n_samples": n_samples,
                "tumor_avg_entropy": float(tumor_avg_entropy),
                "npz_path": save_path,
            }, f, ensure_ascii=False, indent=2)
    print(f"人员3 交付: {len(dataset)} 个样本的 mc_predict 输出已保存至 {output_dir}")
    return output_dir


def load_mc_outputs_from_npz(npz_path):
    """人员4 使用：从 npz 读取全部字段。返回 dict {mean, variance, entropy, pred, label}"""
    data = np.load(npz_path)
    return {
        "mean": data["mean"],
        "variance": data["variance"],
        "entropy": data["entropy"],
        "pred": data["pred"],
        "label": data["label"],
    }


In [ ]:
# %% ========================= 模块10 推理 + 评估（每个实验调用一次） =========================
import sys, os, csv
sys.path.insert(0, os.getcwd())          # 保证能 import src
from src.metrics.unified_metrics import calculate_all_metrics


@torch.no_grad()
def predict_single(model, img, device):
    """img: (1, H, W) tensor -> pred: (H, W) numpy数组（类别索引）"""
    img = img.unsqueeze(0).to(device)
    logits = model(img)
    pred = torch.argmax(logits, dim=1).squeeze(0)
    return pred.cpu().numpy()


def get_sample_predictions(model, dataset, device, n_samples=4, seed=SEED):
    """从数据集中随机抽样n_samples个切片，返回(原图, 原始mask, 预测mask)三元组列表"""
    rng = random.Random(seed)
    indices = rng.sample(range(len(dataset)), min(n_samples, len(dataset)))
    samples = []
    for idx in indices:
        img, mask = dataset[idx]
        pred = predict_single(model, img, device)
        samples.append((img.squeeze(0).numpy(), mask.numpy(), pred))
    return samples


def run_mc_and_eval(cfg, model, test_dataset, test_loader):
    """对一个实验：落盘 samples/（契约 A），批量评估并写 evaluation_metrics.csv，返回 (sample_predictions, res)。"""
    model.eval()

    # 1) 贝叶斯 mc_predict 输出（契约 A）落盘到 实验目录/samples/
    samples_dir = os.path.join(cfg.OUTPUT_DIR, "samples")
    save_mc_predict_outputs(model, test_dataset, samples_dir, n_samples=cfg.MC_N_SAMPLES, device=cfg.DEVICE)

    # 2) 批量推理得到 3 类预测（0=背景 1=肾脏 2=肿瘤）
    model.eval()  # MC 结束后恢复 eval 模式，保证 BatchNorm 统计一致
    pred_all, target_all = [], []
    with torch.no_grad():
        for imgs, masks in test_loader:
            logits = model(imgs.to(cfg.DEVICE))
            pred_all.append(torch.argmax(logits, dim=1).cpu().numpy())
            target_all.append(masks.numpy())
    pred_all = np.concatenate(pred_all)       # (N,H,W) 0/1/2
    target_all = np.concatenate(target_all)

    # 3) 统一指标库是"二值掩膜"口径：肿瘤类(class 2)单独二值化
    pred_tumor = (pred_all == 2).astype(np.float32)
    target_tumor = (target_all == 2).astype(np.float32)

    # 4) 熵图：从 samples/ 读真实熵（sample_%04d.npz，与 test_loader 顺序一致）；缺失则回退全 0 占位
    entropy_list, missing = [], False
    for idx in range(len(test_dataset)):
        npz_path = os.path.join(samples_dir, "sample_%04d.npz" % (idx + 1))
        if os.path.isfile(npz_path):
            entropy_list.append(np.load(npz_path)["entropy"])
        else:
            missing = True
            break
    if (not missing) and entropy_list:
        entropy_map = np.stack(entropy_list).astype(np.float32)   # (N,H,W)
        print("已从 samples/ 读取真实熵图（契约 A）")
    else:
        entropy_map = np.zeros_like(pred_tumor)
        print("未找到 samples/ 熵图，使用全 0 占位")

    # 5) 统一指标
    tumor_mask = target_tumor
    res = calculate_all_metrics(pred_tumor, target_tumor, entropy_map, tumor_mask, reject_threshold=0.5)
    print("==== %s 测试集肿瘤分割评估（统一指标库）====" % os.path.basename(cfg.OUTPUT_DIR))
    for k, v in res.items():
        print(f"{k:22s}: {v}")

    # 6) 导出标准表格
    csv_path = os.path.join(cfg.OUTPUT_DIR, "evaluation_metrics.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(res.keys()))
        w.writeheader()
        w.writerow(res)
    print("已保存:", csv_path)

    # 7) 抽样可视化用
    sample_predictions = get_sample_predictions(model, test_dataset, cfg.DEVICE, n_samples=4, seed=cfg.SEED)
    print(f"已从测试集随机抽取 {len(sample_predictions)} 个样本完成推理")
    return sample_predictions, res


In [ ]:
# %% ========================= 模块11 Visualization =========================
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

MASK_CMAP = ListedColormap(["black", "yellow", "red"])  # 0=背景, 1=肾脏, 2=肿瘤


def plot_training_curves(history, save_path=None):
    """绘制训练/验证曲线；验证集为空时只画训练部分（避免空/NaN 数据导致报错）。"""
    epochs = list(range(1, len(history["train_loss"]) + 1))
    has_val = bool(history.get("val_loss")) and len(history["val_loss"]) == len(epochs)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    axes[0, 0].plot(epochs, history["train_loss"], "r-", label="Training Loss")
    if has_val:
        axes[0, 0].plot(epochs, history["val_loss"], "g-", label="Validation Loss")
        best_ep = int(np.argmin(history["val_loss"])) + 1
        axes[0, 0].scatter([best_ep], [history["val_loss"][best_ep - 1]], color="blue",
                           zorder=5, label=f"best epoch= {best_ep}")
    axes[0, 0].set_title("Training and Validation Loss")
    axes[0, 0].set_xlabel("Epochs")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()

    if has_val:
        axes[0, 1].plot(epochs, history["val_dice"], "g-", label="Validation Dice")
        best_ep = int(np.argmax(history["val_dice"])) + 1
        axes[0, 1].scatter([best_ep], [history["val_dice"][best_ep - 1]], color="blue",
                           zorder=5, label=f"best epoch= {best_ep}")
        axes[0, 1].legend()
    axes[0, 1].set_title("Validation Dice Coefficient")
    axes[0, 1].set_xlabel("Epochs")
    axes[0, 1].set_ylabel("Dice")

    if has_val:
        axes[1, 0].plot(epochs, history["val_iou"], "g-", label="Validation IoU")
        best_ep = int(np.argmax(history["val_iou"])) + 1
        axes[1, 0].scatter([best_ep], [history["val_iou"][best_ep - 1]], color="blue",
                           zorder=5, label=f"best epoch= {best_ep}")
        axes[1, 0].legend()
    axes[1, 0].set_title("Validation IoU Coefficient")
    axes[1, 0].set_xlabel("Epochs")
    axes[1, 0].set_ylabel("IoU")

    axes[1, 1].plot(epochs, history["train_loss"], "r-", label="Training Loss")
    best_ep = int(np.argmin(history["train_loss"])) + 1
    axes[1, 1].scatter([best_ep], [history["train_loss"][best_ep - 1]], color="blue",
                       zorder=5, label=f"best epoch= {best_ep}")
    axes[1, 1].set_title("Training Loss")
    axes[1, 1].set_xlabel("Epochs")
    axes[1, 1].set_ylabel("Loss")
    axes[1, 1].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()


def plot_prediction_samples(samples, save_path=None):
    """每行: Original Image | Original Mask | Prediction（与参考模型格式一致）"""
    n = len(samples)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i, (img, mask, pred) in enumerate(samples):
        axes[i, 0].imshow(img, cmap="gray")
        axes[i, 0].set_title("Original Image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(mask, cmap=MASK_CMAP, vmin=0, vmax=2)
        axes[i, 1].set_title("Original Mask")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(pred, cmap=MASK_CMAP, vmin=0, vmax=2)
        axes[i, 2].set_title("Prediction")
        axes[i, 2].axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()


In [ ]:
# ============ 模块12 产物落盘：写 metrics.json（契约 D，供人员4 统计检验） ============
# 每个实验目录：manifest.json + best_unet.pth + 训练曲线 + evaluation_metrics.csv + metrics.json + samples/。
def write_metrics_json(cfg, history, train_cases, val_cases, test_cases, res):
    metrics_out = {
        "experiment": os.path.basename(cfg.EXPERIMENT_DIR) if cfg.EXPERIMENT_DIR else "default",
        "manifest": cfg.EXPERIMENT_MANIFEST,
        "seed": cfg.SEED,
        "epochs": cfg.EPOCHS,
        "batch_size": cfg.BATCH_SIZE,
        "img_size": cfg.IMG_SIZE,
        "n_train_cases": len(train_cases),
        "n_val_cases": len(val_cases),
        "n_test_cases": len(test_cases),
        "best_val_dice": round(max(history["val_dice"]), 4) if history.get("val_dice") else None,
        "best_val_iou": round(max(history["val_iou"]), 4) if history.get("val_iou") else None,
        "metrics": res,
        "notes": "metrics.json 为契约 D 统一产物，供人员4 统计检验；熵相关指标已接入 mc_predict 真实输出（契约 A）。",
    }
    metrics_path = os.path.join(cfg.OUTPUT_DIR, "metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics_out, f, ensure_ascii=False, indent=2)
    print("已保存:", metrics_path)
    return metrics_path


In [ ]:
# ============ 模块13 一键三实验：data10pct / data20pct / datafull ============
# 一次 Run All 顺序执行三个实验，每个实验独立完成：
#   实验目录 = experiments/<data10pct|data20pct|datafull>/
#   产物     = manifest.json + best_unet.pth + training_curves.png + prediction_samples.png
#            + evaluation_metrics.csv + metrics.json + samples/(每样本 npz+json)
EXPERIMENTS = ["data10pct", "data20pct", "datafull"]
results_summary = {}

for exp_name in EXPERIMENTS:
    print("\n" + "=" * 60)
    print(">>> 开始实验:", exp_name)
    print("=" * 60)

    env = setup_experiment(exp_name, train_transform=train_transform)
    train_cases = env["train_cases"]
    val_cases = env["val_cases"]
    test_cases = env["test_cases"]

    model = build_model(cfg)
    criterion, optimizer, scheduler = build_optimizer(model, cfg)

    history, best_val_dice, best_ckpt_path = train_experiment(
        cfg, model, env["train_loader"], env["val_loader"], criterion, optimizer, scheduler)

    sample_predictions, res = run_mc_and_eval(cfg, model, env["test_dataset"], env["test_loader"])

    plot_training_curves(history, save_path=os.path.join(cfg.OUTPUT_DIR, "training_curves.png"))
    if sample_predictions:
        plot_prediction_samples(sample_predictions, save_path=os.path.join(cfg.OUTPUT_DIR, "prediction_samples.png"))
    plt.close("all")

    write_metrics_json(cfg, history, train_cases, val_cases, test_cases, res)

    results_summary[exp_name] = {
        "dice": float(res["dice"]), "iou": float(res["iou"]),
        "n_train_cases": len(train_cases), "n_test_cases": len(test_cases),
    }
    print(">>> 实验完成:", exp_name, "->", cfg.OUTPUT_DIR)

print("\n" + "=" * 60)
print("全部实验完成，汇总：")
print("=" * 60)
for k, v in results_summary.items():
    print("  %-10s dice=%.4f iou=%.4f n_train=%d n_test=%d" % (
        k, v["dice"], v["iou"], v["n_train_cases"], v["n_test_cases"]))
